# Build a Deep Reference Template
accomplish this by median-stacking the aligend epochs.
A median stack is quieter (~1.5-2x less noise) and transient-free (movers appear
in only 1 epoch -> voted out), giving a proper static-sky reference.


## Cell 1: setup + load+ stack + verify

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))
import config 

import glob
import numpy as np 
import matplotlib.pyplot as plt 
from astropy.io import fits
from astropy.visualization import ZScaleInterval

ALIGNED = "../ztfdata/aligned"
SCIENCE_NAME = "ztf_20180322273264"

paths = sorted(glob.glob(os.path.join(ALIGNED, "*sciimg.fits")))
template_paths = [p for p in paths if SCIENCE_NAME not in os.path.basename(p)]
print(f"{len(template_paths)} frames go into the deep template:")

for p in template_paths:
    print(" ", os.path.basename(p))

stack = np.array([fits.getdata(p).astype(float) for p in template_paths])
print("\nstack shape:", stack.shape, "  NaNs in stack:", np.isnan(stack).sum())


In [ ]:
# median across the epoch axis -> the deep template. nanmedian fills reproject-edge
# NaNs from the other frames, so the template comes out NaN-free.
deep_template = np.nanmedian(stack, axis=0)
print("deep template:", deep_template.shape, "  NaNs:", np.isnan(deep_template).sum())

# prove it's quieter: compare noise in a blank patch, single frame vs deep template
c = 1540
patch = (slice(c, c + 200), slice(c, c + 200))
print(f"noise (std) single ref frame: {np.nanstd(stack[1][patch]):.2f}")
print(f"noise (std) deep template:    {np.nanstd(deep_template[patch]):.2f}  (lower = quieter)")

# save it (carry the science frame's WCS so the diff lines up — all share the grid)
sci_hdr = fits.getheader(glob.glob(
    os.path.join(ALIGNED, f"*{SCIENCE_NAME}*sciimg.fits"))[0])
out = "../ztfdata/aligned/deep_template.fits"
fits.writeto(out, deep_template.astype(np.float32), header=sci_hdr, overwrite=True)
print("saved ->", out)


In [ ]:
# Rung 4 — build our own difference: science - deep_template, on the central 1000x1000.
from astropy.nddata import Cutout2D
from scipy.ndimage import gaussian_filter

SCI_PATH = glob.glob(os.path.join(ALIGNED, f"*{SCIENCE_NAME}*sciimg.fits"))[0]
sci_full = fits.getdata(SCI_PATH).astype(float)
tmpl_full = fits.getdata("../ztfdata/aligned/deep_template.fits").astype(float)

# same central 1000x1000 crop Stage 3 used (so catalog x/y centroids still line up)
center = (sci_full.shape[1] // 2, sci_full.shape[0] // 2)
sci_1000  = Cutout2D(sci_full,  center, 1000).data
tmpl_1000 = Cutout2D(tmpl_full, center, 1000).data
print("cropped:", sci_1000.shape, tmpl_1000.shape)


In [ ]:
# PSF-match: the science frame (seeing 1.856) is BLURRIER than the template epochs,
# but the deep median mixes seeings; blur the sharper of the pair to match. The gap
# is tiny here (Rung-2 check), so this barely changes the result -- included for rigor.
SEE_SCI = 1.856   # from the raw sci header
SEE_TMPL = 1.752  # sharpest template epoch (median is ~this or blurrier)
PIXSCALE = 1.012
fwhm = lambda s: s / PIXSCALE
if SEE_TMPL < SEE_SCI:                       # template sharper -> blur template to sci
    sig = np.sqrt(max(fwhm(SEE_SCI)**2 - fwhm(SEE_TMPL)**2, 0)) / 2.355
    tmpl_m = gaussian_filter(tmpl_1000, sig)
    print(f"blurred template by sigma={sig:.2f}px to match science seeing")
else:
    tmpl_m = tmpl_1000

diff_deep = sci_1000 - tmpl_m

# verify the dipole is gone at src_p1_005 (catalog centroid in the 1000-frame)
XC, YC = 339.4519, 252.8579
w = Cutout2D(diff_deep, (XC, YC), 41, mode="partial", fill_value=0).data
print(f"\nsrc_p1_005 window: max={w.max():.1f}  min={w.min():.1f}  span={w.max()-w.min():.1f}")
print("  (ZTF official was max 181 / min -247 / span 428 -- a strong dipole)")
print("  (want: min near 0 or positive, small span = clean point, no dipole)")

# save our clean diff for Rung 5 (re-cut cutouts from THIS)
fits.writeto("../ztfdata/difference/ztf_20180322273264_deep_diff.fits",
             diff_deep.astype(np.float32), overwrite=True)
print("\nsaved -> ../ztfdata/difference/ztf_20180322273264_deep_diff.fits")
